[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/05.%20The%20Single%20Truck%20Gate%20Entry%20Analysis/P5-Tier-1_executed.ipynb)

---


# 5. The Single Truck Gate Entry Analysis

## Tier 1 — Mathematical Formulation

### Goal and scope
This notebook formulates the allocation of a single stream of arriving trucks among gate booths as a linear program. The decision is how much flow, measured in trucks per hour, to assign to each gate. The model minimizes a linear time-related cost subject to flow conservation and gate-capacity constraints.

The linear program does **not** calculate congestion-dependent queueing costs in its objective. Instead, it uses each gate's service time as a fixed per-truck cost proxy. Queueing waiting times are calculated separately after optimization as performance diagnostics. This distinction matters: a linear objective with fixed per-truck costs cannot represent the way a gate's waiting time rises as its utilization approaches capacity.

### Inputs and outputs
- **Inputs:** number of gates, total truck arrival rate, each gate's service rate and service time, and the time-related cost rate.
- **Optimization output:** the flow assigned to each gate and the minimum modeled cost per hour.
- **Performance diagnostics:** gate utilization and estimated queue waiting time under an M/M/1 approximation.

### Modeling assumptions
- Truck flow is divisible, so decision variables may be non-integers.
- Every arriving truck can be assigned to any gate; routing restrictions and gate-opening decisions are outside the model.
- Arrival and service rates are deterministic inputs to the linear program.
- The time-related cost rate is charged against service time to form a fixed cost per truck. It is a proxy, not a congestion-dependent waiting-cost calculation.
- The queueing diagnostic treats each gate as an M/M/1 queue: arrivals are Poisson, service times are exponentially distributed, and the gate is stable only when its assigned arrival rate is strictly less than its service rate.

### Concrete example
The example has two gates and a total arrival rate of 20 trucks per hour:

| Quantity | Gate 1 | Gate 2 |
|:--|--:|--:|
| Service rate, $\\mu_i$ (trucks/hour) | 12 | 15 |
| Service time, $s_i$ (minutes/truck) | 5 | 4 |

The time-related cost rate is $2 per minute of service time. The service times agree with the stated rates: $60/12=5$ minutes per truck and $60/15=4$ minutes per truck.

### Network-flow interpretation
Imagine a source node supplying the arriving trucks, one intermediate node for each gate, and a sink node collecting processed trucks. The flow from the source through gate $i$ is the decision variable $x_i$. Flow conservation sends all arrivals through the gates, while each gate arc has capacity $\\mu_i$. This is a single-period, single-commodity flow-allocation model.

### Linear-programming formulation
Let $G=\\{1,\\ldots,n\\}$ be the set of gates. For gate $i$, let $x_i$ be its assigned flow in trucks per hour, $\\mu_i$ its capacity in trucks per hour, and $s_i$ its service time in minutes per truck. Let $\\lambda$ be the total arrival rate in trucks per hour and $w$ the time-related cost rate in dollars per minute. Charging this rate against service time gives the per-truck coefficient $c_i=w s_i$, measured in dollars per truck. The objective and constraints are:

$$
\\begin{aligned}
\\min_{x_1,\\ldots,x_n} \\quad & \\sum_{i\\in G} c_i x_i
&& \\text{(modeled cost per hour)}\\\\
\\text{subject to} \\quad & \\sum_{i\\in G} x_i = \\lambda
&& \\text{(flow conservation)}\\\\
& 0 \\leq x_i \\leq \\mu_i, \\quad i\\in G
&& \\text{(non-negativity and gate capacity).}
\\end{aligned}
$$

The product $c_i x_i$ has units of dollars per hour. Because $c_i$ is fixed for each gate, the objective is linear. The code builds the coefficient list in `formulate_optimization_problem` and passes the equality, capacity, and non-negativity constraints to `linprog` in `solve_optimal_allocation`.

### Queueing diagnostic
After the allocation is found, let $\\rho_i=x_i/\\mu_i$ be gate $i$'s utilization. Under the M/M/1 assumption, the expected queue waiting time (excluding service) is

$$
W_{q,i} = \\frac{\\rho_i}{1-\\rho_i}\\,\\frac{1}{\\mu_i}
= \\frac{x_i}{\\mu_i(\\mu_i-x_i)}, \\qquad 0\\leq x_i<\\mu_i.
$$

Here $1/\\mu_i$ is the mean service time in hours. At or above full utilization, a steady-state M/M/1 queue has no finite expected waiting time; the code reports infinity for that diagnostic. This queueing equation is not part of the linear-programming objective. In particular, the optimizer can assign a gate its full capacity when that gate has the lower fixed cost, so its queueing diagnostic may be infinite.

For positive total flow, the system-wide diagnostic is the flow-weighted average $\\sum_i x_i W_{q,i}/\\sum_i x_i$, provided all gates carrying positive flow have finite waiting times. If any such gate is unstable, the system-wide diagnostic is infinite.

### What to examine
- Whether the assigned flows sum to the arrival rate and remain within gate capacities.
- How the fixed service-time cost coefficients shape the allocation.
- How utilization affects the separate queueing diagnostic, especially near capacity.
- Why an allocation that minimizes the stated linear objective is not necessarily an allocation that minimizes congestion-dependent waiting time.

In [ ]:
# These libraries support the linear program, its numerical checks, and result plots.
import numpy as np
from scipy.optimize import linprog
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

In [ ]:
class GateEntryNetworkFlow:
    """Linear flow-allocation model for a single truck-entry stream."""

    def __init__(self, num_gates, arrival_rate, service_rates, service_times, waiting_cost):
        """Store the gate data and check that the model inputs are consistent.

        Parameters
        ----------
        num_gates : int
            Number of gate booths.
        arrival_rate : float
            Total truck arrival rate, in trucks per hour.
        service_rates : sequence of float
            Gate service rates, in trucks per hour.
        service_times : sequence of float
            Mean service times, in minutes per truck.
        waiting_cost : float
            Time-related cost rate, in dollars per minute of service time.
            It defines a fixed per-truck cost proxy; it is not a queueing-cost
            term in the linear-programming objective.
        """
        self.num_gates = num_gates
        self.arrival_rate = float(arrival_rate)
        self.service_rates = np.asarray(service_rates, dtype=float)
        self.service_times = np.asarray(service_times, dtype=float)
        self.waiting_cost = float(waiting_cost)

        if not isinstance(num_gates, int) or num_gates <= 0:
            raise ValueError("num_gates must be a positive integer.")
        if self.service_rates.shape != (num_gates,):
            raise ValueError("service_rates must contain one value per gate.")
        if self.service_times.shape != (num_gates,):
            raise ValueError("service_times must contain one value per gate.")
        if self.arrival_rate < 0 or self.waiting_cost < 0:
            raise ValueError("arrival_rate and waiting_cost must be non-negative.")
        if np.any(self.service_rates <= 0) or np.any(self.service_times <= 0):
            raise ValueError("All service rates and service times must be positive.")

        self.optimal_flows = None
        self.total_cost = None

    def formulate_optimization_problem(self):
        """Return fixed per-truck cost coefficients for the linear program.

        Multiplying dollars per minute by minutes per truck gives dollars
        per truck. Multiplication by flow in trucks per hour then gives
        the modeled cost in dollars per hour.
        """
        cost_coefficients = self.waiting_cost * self.service_times
        return cost_coefficients.tolist()

    def solve_optimal_allocation(self):
        """Minimize modeled cost subject to flow conservation and capacities."""
        if self.arrival_rate > np.sum(self.service_rates):
            print("Optimization failed: total arrival rate exceeds total gate capacity.")
            return False

        cost_coeffs = self.formulate_optimization_problem()

        # The equality sends every arriving truck through one of the gates;
        # the identity matrix applies each gate's own capacity limit.
        A_eq = [np.ones(self.num_gates)]
        b_eq = [self.arrival_rate]
        A_ub = np.eye(self.num_gates)
        b_ub = self.service_rates
        bounds = [(0, None) for _ in range(self.num_gates)]

        result = linprog(
            c=cost_coeffs,
            A_ub=A_ub,
            b_ub=b_ub,
            A_eq=A_eq,
            b_eq=b_eq,
            bounds=bounds,
            method="highs",
        )

        if result.success:
            self.optimal_flows = result.x
            self.total_cost = result.fun
            return True

        print(f"Optimization failed: {result.message}")
        return False

    def calculate_performance_metrics(self):
        """Calculate utilization and M/M/1 queue-wait diagnostics."""
        if self.optimal_flows is None:
            return None

        gate_utilizations = self.optimal_flows / self.service_rates
        avg_waiting_times = []

        for flow, utilization, service_rate in zip(
            self.optimal_flows, gate_utilizations, self.service_rates
        ):
            # A queue at full utilization has no finite steady-state M/M/1 wait.
            if utilization < 1.0 and not np.isclose(
                utilization, 1.0, rtol=0.0, atol=1e-9
            ):
                waiting_time = utilization / (1.0 - utilization) / service_rate
            else:
                waiting_time = float("inf")
            avg_waiting_times.append(waiting_time)

        total_flow = float(np.sum(self.optimal_flows))
        positive_flow_indices = self.optimal_flows > 1e-12

        if total_flow == 0:
            system_avg_waiting = 0.0
        elif any(
            not np.isfinite(avg_waiting_times[i])
            for i in range(self.num_gates)
            if positive_flow_indices[i]
        ):
            system_avg_waiting = float("inf")
        else:
            weighted_waiting = sum(
                self.optimal_flows[i] * avg_waiting_times[i]
                for i in range(self.num_gates)
            )
            system_avg_waiting = weighted_waiting / total_flow

        return {
            "gate_flows": self.optimal_flows.copy(),
            "gate_utilizations": gate_utilizations.tolist(),
            "avg_waiting_times": avg_waiting_times,
            "system_avg_waiting": system_avg_waiting,
            "total_cost": self.total_cost,
            "mean_gate_utilization": float(np.mean(gate_utilizations)),
            "aggregate_capacity_utilization": total_flow / float(np.sum(self.service_rates)),
        }

    def visualize_results(self, metrics):
        """Plot flows, utilization, queue waits, and a concise summary."""
        if metrics is None:
            return

        gates = [f"Gate {i + 1}" for i in range(self.num_gates)]
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(
            "Gate Entry Flow Allocation and Performance Diagnostics",
            fontsize=16,
            fontweight="bold",
        )

        axes[0, 0].bar(gates, metrics["gate_flows"], color="skyblue", alpha=0.8)
        axes[0, 0].set_title("Truck Flow Allocation")
        axes[0, 0].set_ylabel("Flow (trucks per hour)")
        axes[0, 0].grid(True, axis="y", alpha=0.3)
        for i, value in enumerate(metrics["gate_flows"]):
            axes[0, 0].text(i, value, f"{value:.2f}", ha="center", va="bottom")

        utilization_percent = 100 * np.asarray(metrics["gate_utilizations"])
        axes[0, 1].bar(gates, utilization_percent, color="lightgreen", alpha=0.8)
        axes[0, 1].set_title("Gate Utilization")
        axes[0, 1].set_ylabel("Utilization (percent)")
        axes[0, 1].set_ylim(0, 110)
        axes[0, 1].grid(True, axis="y", alpha=0.3)
        for i, value in enumerate(utilization_percent):
            axes[0, 1].text(i, value + 2, f"{value:.1f}%", ha="center", va="bottom")

        finite_waits = [
            value if np.isfinite(value) else 0.0
            for value in metrics["avg_waiting_times"]
        ]
        axes[1, 0].bar(gates, finite_waits, color="salmon", alpha=0.8)
        axes[1, 0].set_title("M/M/1 Queue Waiting-Time Diagnostic")
        axes[1, 0].set_ylabel("Queue waiting time (minutes)")
        axes[1, 0].grid(True, axis="y", alpha=0.3)
        for i, value in enumerate(metrics["avg_waiting_times"]):
            if np.isfinite(value):
                axes[1, 0].text(i, value, f"{value * 60:.2f} min", ha="center", va="bottom")
            else:
                axes[1, 0].text(
                    i,
                    0,
                    "Unstable\n(infinite wait)",
                    ha="center",
                    va="bottom",
                    color="darkred",
                )

        system_wait = metrics["system_avg_waiting"]
        system_wait_text = (
            "infinite (at least one loaded gate is unstable)"
            if not np.isfinite(system_wait)
            else f"{system_wait * 60:.2f} minutes per truck"
        )
        summary_text = (
            f"Modeled cost: ${metrics['total_cost']:.2f} per hour\n"
            f"Flow-weighted queue wait: {system_wait_text}\n"
            f"Mean gate utilization: {metrics['mean_gate_utilization'] * 100:.1f}%\n"
            f"Aggregate capacity utilization: "
            f"{metrics['aggregate_capacity_utilization'] * 100:.1f}%"
        )
        axes[1, 1].text(
            0.05,
            0.5,
            summary_text,
            fontsize=11,
            verticalalignment="center",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
        axes[1, 1].set_title("Model and Diagnostic Summary")
        axes[1, 1].axis("off")

        plt.tight_layout()
        plt.show()

    def print_detailed_results(self, metrics):
        """Print the allocation and distinguish modeled cost from queue diagnostics."""
        if metrics is None:
            return

        print("=" * 64)
        print("GATE ENTRY FLOW ALLOCATION RESULTS")
        print("=" * 64)
        print("\nProblem parameters:")
        print(f"  Arrival rate: {self.arrival_rate:.2f} trucks per hour")
        print(f"  Number of gates: {self.num_gates}")
        print(f"  Time-related cost rate: ${self.waiting_cost:.2f} per minute")

        print("\nOptimal flow allocation:")
        for i in range(self.num_gates):
            print(
                f"  Gate {i + 1}: {metrics['gate_flows'][i]:.2f} trucks per hour "
                f"({metrics['gate_utilizations'][i] * 100:.1f}% utilization)"
            )

        print("\nModel objective and system summaries:")
        print(f"  Modeled time-related cost: ${metrics['total_cost']:.2f} per hour")
        if np.isfinite(metrics["system_avg_waiting"]):
            print(
                "  Flow-weighted M/M/1 queue wait: "
                f"{metrics['system_avg_waiting'] * 60:.2f} minutes per truck"
            )
        else:
            print("  Flow-weighted M/M/1 queue wait: infinite (a loaded gate is unstable)")
        print(
            "  Mean utilization across gates: "
            f"{metrics['mean_gate_utilization'] * 100:.1f}%"
        )
        print(
            "  Aggregate capacity utilization: "
            f"{metrics['aggregate_capacity_utilization'] * 100:.1f}%"
        )

        print("\nGate-level queueing diagnostics:")
        for i in range(self.num_gates):
            waiting_time = metrics["avg_waiting_times"][i]
            waiting_text = (
                "infinite (unstable at this utilization)"
                if not np.isfinite(waiting_time)
                else f"{waiting_time * 60:.2f} minutes"
            )
            print(f"  Gate {i + 1}:")
            print(f"    Flow: {metrics['gate_flows'][i]:.2f} trucks per hour")
            print(f"    Service rate: {self.service_rates[i]:.2f} trucks per hour")
            print(f"    Service time: {self.service_times[i]:.2f} minutes per truck")
            print(f"    Queue waiting-time diagnostic: {waiting_text}")

In [ ]:
print("Concrete Example: Simplified Gate System with 2 Gates")
print("=" * 56)

# The example parameters define total demand, each gate's capacity and
# service duration, and the fixed time-related cost rate.
num_gates = 2
arrival_rate = 20  # Trucks per hour
service_rates = [12, 15]  # Trucks per hour for each gate
service_times = [5, 4]  # Minutes per truck for each gate
waiting_cost = 2  # Dollars per minute of service time

gate_model = GateEntryNetworkFlow(
    num_gates=num_gates,
    arrival_rate=arrival_rate,
    service_rates=service_rates,
    service_times=service_times,
    waiting_cost=waiting_cost,
)

success = gate_model.solve_optimal_allocation()

if success:
    metrics = gate_model.calculate_performance_metrics()
    gate_model.print_detailed_results(metrics)
    gate_model.visualize_results(metrics)
else:
    print("The gate allocation problem has no feasible solution.")

In [ ]:
# Check that the reported allocation satisfies the LP constraints and that
# its objective agrees with the cost coefficients used by the solver.
print("\n" + "=" * 56)
print("MODEL FEASIBILITY AND OBJECTIVE CHECKS")
print("=" * 56)

if success and metrics is not None:
    flows = np.asarray(metrics["gate_flows"], dtype=float)
    cost_coefficients = np.asarray(
        gate_model.formulate_optimization_problem(), dtype=float
    )

    assert np.isclose(np.sum(flows), arrival_rate, rtol=1e-9, atol=1e-9)
    assert np.all(flows >= -1e-9)
    assert np.all(flows <= np.asarray(service_rates) + 1e-9)
    assert np.isclose(
        np.dot(cost_coefficients, flows),
        metrics["total_cost"],
        rtol=1e-9,
        atol=1e-9,
    )

    print("Flow conservation: satisfied")
    print("Non-negativity and gate capacities: satisfied")
    print("Reported objective equals the cost-weighted flow total: satisfied")
    print(
        "The queueing diagnostics are separate from the linear-programming "
        "objective; an infinite diagnostic indicates that a loaded gate is "
        "at or numerically indistinguishable from full utilization."
    )
else:
    print("Checks were not run because the optimization did not produce a solution.")

## Interpreting the formulation

### What the linear program guarantees
When the problem is feasible, `linprog` finds an allocation that minimizes the stated linear cost under the flow-conservation and capacity constraints. Feasibility requires that total demand not exceed the sum of gate capacities. The objective is the sum of fixed per-truck service-time costs multiplied by assigned flows, so its unit is dollars per hour.

The coefficients determine the allocation. A gate with a shorter service time has a lower fixed cost per truck and is therefore attractive to the objective. If its capacity is reached, any remaining flow must use other gates. If two or more gates have identical minimum cost, multiple allocations can attain the same optimum; the solver may return any one of them.

### What the queueing diagnostics mean
Utilization is the assigned flow divided by service capacity. A utilization below one is necessary for a stable M/M/1 queue. As utilization approaches one, the estimated queue waiting time increases sharply; at full utilization, the steady-state formula has no finite value. These diagnostics help reveal an operational limitation of an allocation chosen using fixed linear costs: the linear program does not penalize congestion as flow approaches a gate's capacity.

The reported mean gate utilization is the unweighted arithmetic mean across booths. Aggregate capacity utilization is total assigned flow divided by total gate capacity. These are different summaries and neither should be confused with the flow-weighted queue waiting-time diagnostic.

### Strengths, limitations, and appropriate use
**Strengths**
- The objective and constraints are explicit and interpretable.
- The linear program gives an optimum for the stated fixed-cost allocation model.
- Flow and capacity checks make feasibility easy to inspect.
- The separate queueing calculation highlights how utilization relates to waiting under its M/M/1 assumptions.

**Limitations**
- The objective uses a fixed service-time cost proxy, not congestion-dependent waiting cost.
- Deterministic inputs do not represent variability in arrivals, service, or operating conditions.
- The M/M/1 diagnostic relies on queueing assumptions and is not part of the optimization model.
- Routing restrictions, time-varying demand, and other operational details are outside the model's scope.

This formulation is useful for a basic capacity-constrained allocation and for understanding the difference between an optimization objective and a post-solution performance diagnostic. Its results should be interpreted within those modeling assumptions.